# MNIST-family image embeddings: IMSTE and UMAP

Choose dataset, graph, optimizer, and rendering settings, then compare IMSTE with UMAP on the same standardized samples. Multilevel coarsening uses IMSTE graph ranks; by default, PCA of pooled coarse-group features initializes its smallest graph. Both embeddings are displayed with `render_datashader_embedding` to keep dense plots readable.

The full gallery, including class-mixture graph analysis, is archived at `notebooks/archive/high_dim_mst_gallery_full.ipynb`.

This notebook does not cache downloaded datasets or fitted embeddings: every run fetches the raw MNIST variants again and refits IMSTE, gravity, and UMAP.


In [1]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "src" / "high_dim_mst_gallery.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks" / "src"))
    sys.path.insert(0, str(repo_root))

## 1. Select parameters and fit IMSTE

The main IMSTE controls are the number of MSTs, optimizer epochs, gravity strength, and mean-shift iterations. The next code cell sets these four values; the other estimator options use their defaults. Dataset sampling, UMAP, and Datashader controls are listed separately.

The notebook uses the default Euclidean metric, PCA initialization, and multilevel initialization. Gravity requires Euclidean features plus RINND, igraph, and leidenalg.

In [ ]:
from high_dim_mst_gallery import run_high_dim_mst_gallery
from IPython.display import display

# Dataset size
MAX_SAMPLES = 60_000
RANDOM_STATE = 42

# Main IMSTE controls; all other estimator options use their defaults.
IMSTE_N_MSTS = 3
IMSTE_N_EPOCHS = 200
IMSTE_GRAVITY_STRENGTH = 0.2
IMSTE_MEAN_SHIFT_ITERATIONS = 1

# UMAP comparison
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_EPOCHS = 200
UMAP_RECURSIVE_COARSENING_RATIO = 4
REPULSION_STRENGTH = 1

# Datashader display
RENDER_WIDTH = 700
RENDER_HEIGHT = 700
RENDER_MIN_ALPHA = 40
RENDER_SPREAD_PX = 1
RENDER_BACKGROUND = "white"  # Change to "black" to compare contrast.

# Load the image datasets and fit IMSTE. Sample image grids and the helper's
# scatter plots are disabled; the comparison below uses Datashader instead.
gallery = run_high_dim_mst_gallery(
    max_samples=MAX_SAMPLES,
    n_msts=IMSTE_N_MSTS,
    n_epochs=IMSTE_N_EPOCHS,
    gravity_strength=IMSTE_GRAVITY_STRENGTH,
    mean_shift_iterations=IMSTE_MEAN_SHIFT_ITERATIONS,
    random_state=RANDOM_STATE,
    plot_sample_images=False,
    plot_embeddings=False,
)
display(gallery["summary"])
display(gallery["summary"][
    ["dataset", "gravity_neighbors", "gravity_resolution",
     "gravity_strength", "mean_shift_neighbors", "mean_shift_bandwidth_factor",
     "mean_shift_step_size", "mean_shift_iterations", "gravity_regions_packed"]
])
if gallery["load_errors"]:
    display(gallery["load_errors"])


## 2. Fit UMAP to the same samples

The gallery helper standardizes features before IMSTE. This cell applies the same standardization to each raw dataset before fitting UMAP, so both methods receive matching input rows and feature scaling. Gravity uses the same standardized input and never uses UMAP coordinates.


In [ ]:
from time import perf_counter

import umap
from IPython.display import display
from sklearn.preprocessing import StandardScaler

UMAP_RESULTS = {}
UMAP_SECONDS = {}
for dataset_name, (raw_X, labels) in gallery["datasets"].items():
    X_standardized = StandardScaler().fit_transform(raw_X)
    started = perf_counter()
    embedding = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=METRIC,
        init="recursive",
        recursive_coarsening_ratio=UMAP_RECURSIVE_COARSENING_RATIO,
        compatibility_layout=False,
        nearest_neighbor_backend="rinnd",
        random_state=RANDOM_STATE,
        n_epochs=UMAP_EPOCHS,
        repulsion_strength=REPULSION_STRENGTH,
        n_jobs=1,
    ).fit_transform(X_standardized)
    UMAP_RESULTS[dataset_name] = embedding
    UMAP_SECONDS[dataset_name] = perf_counter() - started
    print(f"{dataset_name}: UMAP fit in {UMAP_SECONDS[dataset_name] / 60:.1f} min")

comparison = gallery["summary"][["dataset", "samples", "input_features", "seconds"]].rename(
    columns={"seconds": "IMSTE seconds"}
)
comparison["UMAP seconds"] = comparison["dataset"].map(UMAP_SECONDS)
display(comparison)


## 3. Compare the embeddings with Datashader

Each panel rasterizes all points at the selected image size. Adjust the rendering settings above to change the mark spread, opacity, resolution, or background.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from imste.visualization import render_datashader_embedding

DATASHADER_IMAGES = {}
summary = gallery["summary"].set_index("dataset")
IMSTE_SECONDS = summary["imste_seconds"].to_dict()
GRAVITY_SECONDS = summary["gravity_seconds"].to_dict()
MEAN_SHIFT_SECONDS = summary["mean_shift_seconds"].to_dict()
s = 7
for dataset_name, (_, labels) in gallery["datasets"].items():
    embeddings = {
        "IMSTE · multilevel start": gallery["base_embeddings"][dataset_name],
    }
    if IMSTE_GRAVITY_STRENGTH > 0 or IMSTE_MEAN_SHIFT_ITERATIONS > 0:
        embeddings["IMSTE · gravity → mean shift"] = gallery["embeddings"][dataset_name]
    embeddings["UMAP"] = UMAP_RESULTS[dataset_name]
    fig, axes = plt.subplots(1, len(embeddings), figsize=(9 * len(embeddings), 7), constrained_layout=True, squeeze=False)
    for ax, (method, embedding) in zip(axes.ravel(), embeddings.items()):
        image = render_datashader_embedding(
            embedding,
            labels,
            width=300*s,
            height=250*s,
            min_alpha=RENDER_MIN_ALPHA,
            background=RENDER_BACKGROUND,
            spread_px=3,
        )
        DATASHADER_IMAGES[(dataset_name, method)] = image
        ax.imshow(image.to_pil())
        elapsed = (
            IMSTE_SECONDS[dataset_name] + GRAVITY_SECONDS[dataset_name] + MEAN_SHIFT_SECONDS[dataset_name]
            if method == "IMSTE · gravity → mean shift"
            else IMSTE_SECONDS[dataset_name]
            if method.startswith("IMSTE")
            else UMAP_SECONDS[dataset_name]
        )
        runtime = f"{elapsed / 60:.1f} min" if elapsed >= 60 else f"{elapsed:.1f} s"
        ax.set_title(f"{dataset_name} — {method} ({runtime})")
        ax.set_axis_off()
    display(fig)
    plt.close(fig)
